<a href="https://colab.research.google.com/github/mahdad277/repo1/blob/vector-search/zilliz_image_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip3 install pymilvus==2.5.3
!pip install openai-clip

In [ ]:
import clip
import torch
from PIL import Image

from pymilvus import connections, MilvusClient

client = MilvusClient(
    uri="https://in03-a6d314869a5e4ff.serverless.gcp-us-west1.cloud.zilliz.com",
    token="<put your API token here>",
)

print(client.list_collections())

model, preprocess = clip.load("ViT-B/32")
image318 = preprocess(Image.open("order-318-missing.jpg")).unsqueeze(0)
with torch.no_grad():
    image_features318 = model.encode_image(image318)

['medium_articles', 'images']


In [ ]:
# Convert torch tensor to Python list
vector318 = image_features318[0].tolist()

# Step 3: Insert into collection
data = [
    {
        "vector": vector318,
        "filename": "order-318-missing.jpg",  # optional metadata
    }
]

client.insert(collection_name="images", data=data)


{'insert_count': 1, 'ids': [459906201856645592], 'cost': 1}

In [ ]:
# Optional: Search back to verify
results = client.search(
    collection_name="images",
    data=[vector318],
    output_fields=["filename"],
    limit=2
)

print("Search results:", results)

Search results: data: ["[{'id': 459906201856645592, 'distance': 1.0000001192092896, 'entity': {'filename': 'order-318-missing.jpg'}}]"] , extra_info: {'cost': 6}


In [ ]:
# now that we inserted image 1, create a vector for image 2 and do a search
image384 = preprocess(Image.open("order-384.jpg")).unsqueeze(0)
with torch.no_grad():
    image_features384 = model.encode_image(image384)
vector384 = image_features384[0].tolist()

results = client.search(
    collection_name="images",
    data=[vector384],
    output_fields=["filename"],
    limit=2
)

print("Search results:", results)


Search results: data: ["[{'id': 459906201856645592, 'distance': 0.8823705315589905, 'entity': {'filename': 'order-318-missing.jpg'}}]"] , extra_info: {'cost': 6}
